In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

from itertools import chain
from re import split
from toolz import pipe, unique, compose_left
from toolz.curried import map as mapz, filter as filterz, partial
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize, download
from re import sub

download('punkt_tab')
download('stopwords')

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

## Handle the Data

In [ ]:
def _isNotEmpty(word: str) -> bool:
  return len(word) != 0

_stopwords = stopwords.words('english')
def _isNotStopword(word: str) -> bool:
  return word not in _stopwords

_removeNonAlpha = partial(sub, r'[^a-zA-Z ]+', ' ')

process_text = compose_left(
  str.lower,
  _removeNonAlpha,
  word_tokenize,
  filterz(_isNotEmpty),
  filterz(_isNotStopword),
  lambda filter: " ".join(filter)
)

IS_SPAM = "isSpam"
INPUT_COLUMNS = ["sham", "body"]

def read_sham_tsv(path: str) -> tuple[tf.data.Dataset, int]:
  df = pd.read_csv(
    path,
    sep='\t',
    names=INPUT_COLUMNS,
    dtype="string"
  )

  df["processed"] = df.body.apply(process_text)
  df[IS_SPAM] = df.sham.apply(lambda sham: int(sham == "spam"))

  max_len = df.processed.str.split().str.len().max()

  return pipe(
    df,
    lambda df: (df.processed, df.sham),
    tf.data.Dataset.from_tensor_slices
  ), max_len

train_dataset, train_max = read_sham_tsv(train_file_path)
test_dataset, test_max = read_sham_tsv(test_file_path)
MAX_LEN = 1 << max(train_max, test_max).bit_length()

for x, y in train_dataset.take(1):
  print(x)
  print(y)

tf_layers = tf.keras.layers

text_vector_layer = tf_layers.TextVectorization(output_sequence_length=MAX_LEN)
text_vector_layer.adapt(train_dataset.concatenate(test_dataset))
text_vector_layer.finalize_state()

SHAM_MAPPING = tf.lookup.StaticHashTable(
    initializer=tf.lookup.KeyValueTensorInitializer(
        keys=["ham", "spam"],
        values=[0, 1],
    ),
    default_value=-1,
)

map_to_vector = lambda feature, label: (text_vector_layer(feature), SHAM_MAPPING.lookup(label))
train_dataset = train_dataset.map(map_to_vector)
test_dataset = test_dataset.map(map_to_vector)

for x, y in train_dataset.take(1):
  print(x)
  print(y)


## Build The Model

In [ ]:
model = tf.keras.Sequential([
    tf_layers.Embedding(text_vector_layer.vocabulary_size(), 2 ** 7),
    tf_layers.Bidirectional(tf_layers.LSTM(2 ** 6)),
    tf_layers.Dense(2 ** 5, activation="sigmoid"),
    tf_layers.Dense(2 ** 0, activation="sigmoid")
])
BATCH_SIZE = 32
model.compile(loss='binary_crossentropy',
              optimizer=tf.optimizers.Adam(learning_rate=0.001),
              metrics=['accuracy'])
model.fit(train_dataset.padded_batch(BATCH_SIZE), epochs=5)

## Build Prediction function

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text: str) -> tuple[int, str]:
  input = pipe(
      pred_text,
      process_text,
      text_vector_layer,
      partial(tf.expand_dims, axis=0),
  )
  [[response]] = model.predict(input)
  predicted = "ham" if (response < 0.5) else "spam"
  return (response, predicted)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      print(f"missed answer {msg} expecting {ans}")
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
